# Laboratorio: Arquitecturas Neuronales para NLP — Pipeline End-to-End
## MLP, RNN, LSTM, GRU y BiLSTM para Clasificación de Texto

**Curso:** NLP y Análisis Semántico  
**Prof.:** Francisco Suárez  
**Semana 6 — Sesión de Laboratorio**

---

### Objetivos
1. Construir un pipeline completo de clasificación de texto: desde datos crudos hasta evaluación
2. Implementar y comparar 5 arquitecturas: **MLP**, **RNN**, **LSTM**, **GRU**, **BiLSTM**
3. Entender las diferencias fundamentales entre cada arquitectura completando partes clave del código
4. Visualizar y analizar curvas de entrenamiento, métricas y comportamiento de gradientes

### Instrucciones
- Ejecuta cada celda en orden
- Completa las secciones marcadas con `# TODO`
- Responde las preguntas de reflexión marcadas con **🤔**
- Experimenta cambiando hiperparámetros cuando se indique con **🔬**

### Requisitos
- Google Colab con GPU (Runtime → Change runtime type → T4 GPU)
- Tiempo estimado: 90-120 minutos

---
## Parte 0: Configuración del Entorno

In [1]:
# Verificar GPU disponible
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    print("⚠️ No hay GPU disponible, usando CPU (será más lento)")
    device = torch.device('cpu')

print(f"\nDispositivo seleccionado: {device}")

PyTorch version: 2.10.0+cu126
CUDA disponible: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU

Dispositivo seleccionado: cuda


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
import re

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✅ Todas las librerías cargadas correctamente.")

✅ Todas las librerías cargadas correctamente.


---
## Parte 1: Carga y Exploración de Datos

Usaremos **20 Newsgroups**, un dataset clásico de clasificación de texto.
Seleccionamos 4 categorías para mantener el entrenamiento rápido.

In [3]:
# Categorías seleccionadas — temáticas bien diferenciadas
categories = [
    'sci.space',
    'rec.sport.baseball',
    'comp.graphics',
    'talk.politics.mideast'
]

# Cargar datos
newsgroups_train = fetch_20newsgroups(
    subset='train', categories=categories,
    remove=('headers', 'footers', 'quotes'),  # Eliminar metadata
    random_state=42
)
newsgroups_test = fetch_20newsgroups(
    subset='test', categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print(f"Documentos de entrenamiento: {len(newsgroups_train.data)}")
print(f"Documentos de test: {len(newsgroups_test.data)}")
print(f"\nCategorías: {newsgroups_train.target_names}")
print(f"\nDistribución de clases (train):")
for i, name in enumerate(newsgroups_train.target_names):
    count = (newsgroups_train.target == i).sum()
    print(f"  {name}: {count} documentos")

Documentos de entrenamiento: 2338
Documentos de test: 1556

Categorías: ['comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.politics.mideast']

Distribución de clases (train):
  comp.graphics: 584 documentos
  rec.sport.baseball: 597 documentos
  sci.space: 593 documentos
  talk.politics.mideast: 564 documentos


In [4]:
# Veamos un ejemplo de cada categoría
for i, name in enumerate(newsgroups_train.target_names):
    idx = np.where(newsgroups_train.target == i)[0][0]
    texto = newsgroups_train.data[idx][:300]  # Primeros 300 caracteres
    print(f"\n{'='*60}")
    print(f"📂 Categoría: {name}")
    print(f"{'='*60}")
    print(texto)
    print("...")


📂 Categoría: comp.graphics
Apparently, my editor didn't do what I wanted it to do, so I'll try again.

i'm looking for any programs or code to do simple animation and/or
drawing using fractals in TurboPascal for an IBM
              Thanks in advance
...

📂 Categoría: rec.sport.baseball
[about the infield fly rule]
	Unless he's Deion Sanders, in which case he just heads back to the
dugout and waits for his next base-running-blunder opportunity.
...

📂 Categoría: sci.space


Well, there already is a sulfate process for TiO2 purification.  The
chlorine process is cleaner, however, and for that reason is achieving
dominance in the marketplace.

Most Ti is used in pigment, btw (as the oxide), where it replaced
white lead pigment some decades ago.  Very little is reduced 
...

📂 Categoría: talk.politics.mideast

Proven?  Maybe not.  But it can certainly be verified beyond a reasonable doubt.  This
statement and statements like it are a matter of public record.  Before the Six Day War (1967

---
## Parte 2: Preprocesamiento de Texto

Vamos a construir **dos pipelines** de preprocesamiento:
1. **TF-IDF** → para el modelo MLP (bag-of-words, sin orden)
2. **Secuencias de tokens** → para RNN, LSTM, GRU y BiLSTM (preservan el orden)

### 2.1 Pipeline TF-IDF (para MLP)

In [5]:
# Vectorización TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,   # Vocabulario limitado a 5000 palabras
    stop_words='english',
    min_df=2,            # Ignorar palabras que aparecen en menos de 2 documentos
    max_df=0.95          # Ignorar palabras que aparecen en más del 95% de documentos
)

X_train_tfidf = tfidf.fit_transform(newsgroups_train.data).toarray()
X_test_tfidf = tfidf.transform(newsgroups_test.data).toarray()

y_train = newsgroups_train.target
y_test = newsgroups_test.target

print(f"Forma de X_train (TF-IDF): {X_train_tfidf.shape}")
print(f"Cada documento es un vector de {X_train_tfidf.shape[1]} dimensiones")
print(f"\n🤔 ¿Qué información se PIERDE al usar TF-IDF?")

Forma de X_train (TF-IDF): (2338, 5000)
Cada documento es un vector de 5000 dimensiones

🤔 ¿Qué información se PIERDE al usar TF-IDF?


### 2.2 Pipeline de Secuencias (para modelos recurrentes)

Para RNNs necesitamos preservar el **orden** de las palabras.
Vamos a construir un vocabulario y convertir cada texto en una secuencia de índices.

In [6]:
# Tokenizador simple
def tokenize(text):
    """Tokeniza texto a palabras en minúsculas, solo alfanuméricos."""
    text = text.lower()
    tokens = re.findall(r'\b[a-z]+\b', text)
    return tokens

# Construir vocabulario desde el conjunto de entrenamiento
word_counts = Counter()
for text in newsgroups_train.data:
    word_counts.update(tokenize(text))

print(f"Palabras únicas encontradas: {len(word_counts)}")
print(f"\nTop 15 palabras más frecuentes:")
for word, count in word_counts.most_common(15):
    print(f"  '{word}': {count}")

Palabras únicas encontradas: 27028

Top 15 palabras más frecuentes:
  'the': 25572
  'of': 11735
  'to': 11667
  'and': 11102
  'a': 9282
  'in': 8012
  'i': 6910
  'is': 5717
  'that': 5692
  'it': 4657
  'for': 4445
  'you': 3806
  'on': 3517
  's': 2998
  'was': 2948


In [7]:
# Crear mapeo palabra → índice
VOCAB_SIZE = 5000
PAD_IDX = 0
UNK_IDX = 1

# Tokens especiales + las VOCAB_SIZE-2 palabras más frecuentes
vocab = ['<PAD>', '<UNK>'] + [w for w, _ in word_counts.most_common(VOCAB_SIZE - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}

print(f"Tamaño del vocabulario: {len(vocab)}")
print(f"Índice de '<PAD>': {word2idx['<PAD>']}")
print(f"Índice de '<UNK>': {word2idx['<UNK>']}")
print(f"Índice de 'the':   {word2idx.get('the', 'N/A')}")

Tamaño del vocabulario: 5000
Índice de '<PAD>': 0
Índice de '<UNK>': 1
Índice de 'the':   2


In [ ]:
# TODO: Completa la función que convierte texto a secuencia de índices
def text_to_sequence(text, word2idx, max_len=200):
    """
    Convierte un texto en una secuencia de índices de longitud fija.
    
    Args:
        text: String con el texto
        word2idx: Diccionario palabra → índice
        max_len: Longitud máxima de la secuencia
    
    Returns:
        Lista de enteros de longitud max_len (con padding si es necesario)
    
    Pasos:
        1. Tokenizar el texto
        2. Convertir cada token a su índice (usar UNK_IDX si no está en el vocabulario)
        3. Truncar si la secuencia es más larga que max_len
        4. Hacer padding con PAD_IDX si la secuencia es más corta que max_len
    """
    tokens = tokenize(text)
    
    # TODO: Convertir tokens a índices
    # Pista: usa word2idx.get(token, UNK_IDX) para manejar palabras desconocidas
    indices = ...  # <-- Tu código aquí
    
    # TODO: Truncar a max_len
    indices = ...  # <-- Tu código aquí
    
    # TODO: Padding con PAD_IDX
    padding_length = ...  # <-- Tu código aquí
    indices = ...  # <-- Tu código aquí
    
    return indices


# Verificación
ejemplo = "The quick brown fox jumps over the lazy dog"
seq = text_to_sequence(ejemplo, word2idx, max_len=15)
print(f"Texto: '{ejemplo}'")
print(f"Secuencia: {seq}")
print(f"Longitud: {len(seq)} (debe ser 15)")
print(f"\n¿Los 0s al final son padding? {'Sí ✅' if seq[-1] == 0 else 'No ❌'}")

In [ ]:
# Convertir todos los datos a secuencias
MAX_LEN = 200

X_train_seq = [text_to_sequence(t, word2idx, MAX_LEN) for t in newsgroups_train.data]
X_test_seq = [text_to_sequence(t, word2idx, MAX_LEN) for t in newsgroups_test.data]

# Convertir a tensores de PyTorch
X_train_seq_t = torch.LongTensor(X_train_seq)
X_test_seq_t = torch.LongTensor(X_test_seq)
y_train_t = torch.LongTensor(y_train)
y_test_t = torch.LongTensor(y_test)

# También tensores TF-IDF
X_train_tfidf_t = torch.FloatTensor(X_train_tfidf)
X_test_tfidf_t = torch.FloatTensor(X_test_tfidf)

print(f"Secuencias de entrenamiento: {X_train_seq_t.shape}")
print(f"Ejemplo de secuencia: {X_train_seq_t[0][:20]}...")
print(f"\nTF-IDF de entrenamiento: {X_train_tfidf_t.shape}")

**🤔 Pregunta de reflexión:**
¿Por qué necesitamos dos representaciones diferentes? ¿Qué información captura cada una que la otra no puede?

---
## Parte 3: DataLoaders

Creamos DataLoaders que alimentarán batches a nuestros modelos durante el entrenamiento.

In [ ]:
BATCH_SIZE = 64

# DataLoader para MLP (usa TF-IDF)
train_loader_tfidf = DataLoader(
    TensorDataset(X_train_tfidf_t, y_train_t),
    batch_size=BATCH_SIZE, shuffle=True
)
test_loader_tfidf = DataLoader(
    TensorDataset(X_test_tfidf_t, y_test_t),
    batch_size=BATCH_SIZE
)

# DataLoader para modelos recurrentes (usa secuencias)
train_loader_seq = DataLoader(
    TensorDataset(X_train_seq_t, y_train_t),
    batch_size=BATCH_SIZE, shuffle=True
)
test_loader_seq = DataLoader(
    TensorDataset(X_test_seq_t, y_test_t),
    batch_size=BATCH_SIZE
)

# Verificar un batch
x_batch, y_batch = next(iter(train_loader_seq))
print(f"Batch de secuencias: {x_batch.shape}  (batch_size, max_len)")
print(f"Batch de labels:     {y_batch.shape}")
print(f"\n✅ DataLoaders listos.")

---
## Parte 4: Definición de Arquitecturas

Ahora viene la parte central del laboratorio. Vamos a implementar 5 arquitecturas,
cada una con sus fortalezas y debilidades.

| Modelo | Entrada | ¿Orden? | Parámetros |
|--------|---------|---------|------------|
| MLP | TF-IDF | ❌ | Pocos |
| RNN | Secuencia | ✅ | Pocos |
| LSTM | Secuencia | ✅ | Muchos (4x RNN) |
| GRU | Secuencia | ✅ | Medios (3x RNN) |
| BiLSTM | Secuencia | ✅ (ambas direcciones) | 2x LSTM |

### 4.1 Modelo 1: MLP con TF-IDF (Baseline)

Nuestro punto de referencia. Un MLP opera sobre vectores TF-IDF.

**Recordar:** El MLP no ve el orden de las palabras. "dog bites man" y "man bites dog" producen el mismo vector TF-IDF.

In [ ]:
class MLPClassifier(nn.Module):
    """
    Multi-Layer Perceptron para clasificación de texto.
    Entrada: vector TF-IDF de dimensión fija (5000)
    Arquitectura: Linear → ReLU → Dropout → Linear → ReLU → Dropout → Linear
    """
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),       # 5000 → 128
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), # 128 → 64
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)  # 64 → 4
        )
    
    def forward(self, x):
        # x: (batch_size, input_dim) = (64, 5000)
        return self.net(x)


# Instanciar y verificar
mlp_model = MLPClassifier(
    input_dim=X_train_tfidf.shape[1],
    hidden_dim=128,
    num_classes=len(categories),
    dropout=0.3
)

# Contar parámetros
n_params = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
print(f"MLP - Parámetros entrenables: {n_params:,}")
print(f"\nArquitectura:\n{mlp_model}")

### 4.2 Modelo 2: RNN Vanilla

La RNN procesa la secuencia palabra por palabra, manteniendo un estado oculto $h_t$:

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$

Usamos el **último hidden state** $h_T$ como representación del documento completo.

In [ ]:
class RNNClassifier(nn.Module):
    """
    RNN vanilla para clasificación de texto.
    Entrada: secuencia de índices → Embedding → RNN → Linear
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True  # Formato: (batch, seq_len, features)
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        # x: (batch_size, seq_len) = (64, 200)
        embedded = self.dropout(self.embedding(x))  # (64, 200, embed_dim)
        
        # output: (batch, seq_len, hidden_dim) — todos los h_t
        # h_n:    (1, batch, hidden_dim)       — solo h_T (el último)
        output, h_n = self.rnn(embedded)
        
        # Usamos el último hidden state para clasificar
        h_last = h_n.squeeze(0)  # (batch, hidden_dim)
        logits = self.fc(self.dropout(h_last))
        return logits


# Instanciar
EMBED_DIM = 100
HIDDEN_DIM = 128
NUM_CLASSES = len(categories)

rnn_model = RNNClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
n_params = sum(p.numel() for p in rnn_model.parameters() if p.requires_grad)
print(f"RNN Vanilla - Parámetros entrenables: {n_params:,}")
print(f"\nArquitectura:\n{rnn_model}")

### 4.3 Modelo 3: LSTM

La LSTM añade un **estado de celda** $c_t$ y tres **compuertas** para resolver el problema de gradientes que desaparecen:

- **Forget gate:** $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$
- **Input gate:** $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$  
- **Output gate:** $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$
- **Cell update:** $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$
- **Hidden state:** $h_t = o_t \odot \tanh(c_t)$

In [ ]:
# TODO: Implementa el clasificador LSTM
# Pista: Es casi idéntico al RNNClassifier, pero usa nn.LSTM en vez de nn.RNN
# Recuerda: nn.LSTM retorna (output, (h_n, c_n)) en vez de (output, h_n)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        
        # TODO: Crear la capa LSTM (usa nn.LSTM con batch_first=True)
        self.lstm = ...  # <-- Tu código aquí
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        
        # TODO: Pasar por la LSTM
        # Recuerda: LSTM retorna (output, (h_n, c_n))
        output, (h_n, c_n) = ...  # <-- Tu código aquí
        
        # TODO: Extraer el último hidden state y clasificar
        h_last = ...  # <-- Tu código aquí
        logits = ...  # <-- Tu código aquí
        
        return logits


lstm_model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
n_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"LSTM - Parámetros entrenables: {n_params:,}")
print(f"\n🤔 ¿Por qué tiene ~4x más parámetros que la RNN vanilla?")

### 4.4 Modelo 4: GRU

El GRU simplifica la LSTM con solo **2 compuertas**:

- **Reset gate:** $r_t = \sigma(W_r [h_{t-1}, x_t] + b_r)$
- **Update gate:** $z_t = \sigma(W_z [h_{t-1}, x_t] + b_z)$
- **Estado:** $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

Noten que $(1-z_t) + z_t = 1$ — la memoria siempre se conserva.

In [ ]:
# TODO: Implementa el clasificador GRU
# Pista: Igual que RNN pero con nn.GRU
# nn.GRU retorna (output, h_n) como nn.RNN — NO tiene c_n

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        # TODO: Implementar __init__
        # Necesitas: embedding, gru, dropout, fc
        ...  # <-- Tu código aquí
    
    def forward(self, x):
        # TODO: Implementar forward
        # Pasos: embedding → dropout → GRU → last hidden → dropout → linear
        ...  # <-- Tu código aquí


gru_model = GRUClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
n_params = sum(p.numel() for p in gru_model.parameters() if p.requires_grad)
print(f"GRU - Parámetros entrenables: {n_params:,}")
print(f"\n🤔 ¿Por qué tiene ~3x los parámetros de RNN en vez de 4x como LSTM?")

### 4.5 Modelo 5: BiLSTM (Bidireccional)

Un BiLSTM procesa la secuencia en **ambas direcciones** y concatena los resultados:

```
Forward:  → "The" → "cat" → "sat" → "on" → "the" → "mat" →
Backward: ← "The" ← "cat" ← "sat" ← "on" ← "the" ← "mat" ←
```

El estado final es la concatenación: $[\overrightarrow{h_T} ; \overleftarrow{h_1}]$ con dimensión $2 \times D_h$.

In [ ]:
# TODO: Implementa el clasificador BiLSTM
# Diferencias clave:
# 1. nn.LSTM(..., bidirectional=True)
# 2. h_n tiene forma (2, batch, hidden_dim) — [forward, backward]
# 3. La capa fc recibe 2*hidden_dim (concatenación de ambas direcciones)

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        
        # TODO: Crear LSTM bidireccional
        self.bilstm = ...  # <-- Tu código aquí
        
        self.dropout = nn.Dropout(dropout)
        
        # TODO: ¿Cuál es la dimensión de entrada de fc?
        # Pista: el BiLSTM concatena forward y backward
        self.fc = nn.Linear(..., num_classes)  # <-- Corregir la dimensión
    
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (h_n, c_n) = self.bilstm(embedded)
        
        # h_n: (2, batch, hidden_dim)
        # h_n[0] = último hidden state de la dirección forward
        # h_n[1] = último hidden state de la dirección backward
        
        # TODO: Concatenar ambas direcciones
        h_cat = ...  # <-- Tu código aquí  (resultado: batch, 2*hidden_dim)
        
        logits = self.fc(self.dropout(h_cat))
        return logits


bilstm_model = BiLSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
n_params = sum(p.numel() for p in bilstm_model.parameters() if p.requires_grad)
print(f"BiLSTM - Parámetros entrenables: {n_params:,}")

### 4.6 Resumen de Parámetros

Comparemos el número de parámetros de cada modelo.

In [ ]:
models_info = {
    'MLP': mlp_model,
    'RNN': rnn_model,
    'LSTM': lstm_model,
    'GRU': gru_model,
    'BiLSTM': bilstm_model,
}

print(f"{'Modelo':<12} {'Parámetros':>12} {'Ratio vs RNN':>15}")
print("=" * 42)
rnn_params = sum(p.numel() for p in rnn_model.parameters() if p.requires_grad)
for name, model in models_info.items():
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    ratio = n / rnn_params
    print(f"{name:<12} {n:>12,} {ratio:>14.2f}x")

print(f"\n🤔 ¿La cantidad de parámetros predice directamente la calidad? ¡Vamos a comprobarlo!")

---
## Parte 5: Entrenamiento y Evaluación

Definimos funciones genéricas de entrenamiento y evaluación que funcionan para cualquier modelo.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Entrena el modelo por una época."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Forward
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping (importante para RNNs)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Métricas
        total_loss += loss.item() * X_batch.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += X_batch.size(0)
    
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    """Evalúa el modelo sobre un dataset."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        
        total_loss += loss.item() * X_batch.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += X_batch.size(0)
    
    return total_loss / total, correct / total


print("✅ Funciones de entrenamiento y evaluación definidas.")

In [ ]:
def train_model(model, train_loader, test_loader, device,
                num_epochs=20, lr=0.001, model_name="Modelo"):
    """
    Entrena un modelo y registra métricas por época.
    Incluye early stopping basado en test loss.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': [],
        'epoch_time': []
    }
    
    best_test_acc = 0
    patience = 5
    patience_counter = 0
    
    print(f"\n{'='*60}")
    print(f"Entrenando: {model_name}")
    print(f"{'='*60}")
    
    for epoch in range(num_epochs):
        start = time.time()
        
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        
        elapsed = time.time() - start
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['epoch_time'].append(elapsed)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Época {epoch+1:>3}/{num_epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
                  f"Test Loss: {test_loss:.4f} Acc: {test_acc:.3f} | "
                  f"Tiempo: {elapsed:.1f}s")
        
        # Early stopping
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⏹️ Early stopping en época {epoch+1} (best test acc: {best_test_acc:.3f})")
                break
    
    total_time = sum(history['epoch_time'])
    print(f"\n✅ {model_name}: Best Test Acc = {best_test_acc:.3f} | Tiempo total: {total_time:.1f}s")
    
    return history


print("✅ Función de entrenamiento completa definida.")

### 5.1 Entrenar todos los modelos

Ejecutamos el entrenamiento de los 5 modelos y guardamos sus historiales para comparar.

In [ ]:
NUM_EPOCHS = 20
LR = 0.001

# Diccionario para guardar historiales
all_histories = {}

# 1. MLP (usa TF-IDF loader)
mlp_model = MLPClassifier(X_train_tfidf.shape[1], 128, NUM_CLASSES, dropout=0.3)
all_histories['MLP'] = train_model(
    mlp_model, train_loader_tfidf, test_loader_tfidf, device,
    num_epochs=NUM_EPOCHS, lr=LR, model_name='MLP + TF-IDF'
)

In [ ]:
# 2. RNN Vanilla (usa sequence loader)
rnn_model = RNNClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
all_histories['RNN'] = train_model(
    rnn_model, train_loader_seq, test_loader_seq, device,
    num_epochs=NUM_EPOCHS, lr=LR, model_name='RNN Vanilla'
)

In [ ]:
# 3. LSTM
lstm_model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
all_histories['LSTM'] = train_model(
    lstm_model, train_loader_seq, test_loader_seq, device,
    num_epochs=NUM_EPOCHS, lr=LR, model_name='LSTM'
)

In [ ]:
# 4. GRU
gru_model = GRUClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
all_histories['GRU'] = train_model(
    gru_model, train_loader_seq, test_loader_seq, device,
    num_epochs=NUM_EPOCHS, lr=LR, model_name='GRU'
)

In [ ]:
# 5. BiLSTM
bilstm_model = BiLSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
all_histories['BiLSTM'] = train_model(
    bilstm_model, train_loader_seq, test_loader_seq, device,
    num_epochs=NUM_EPOCHS, lr=LR, model_name='BiLSTM'
)

---
## Parte 6: Comparación y Análisis

Ahora viene lo más importante: **interpretar los resultados**.

### 6.1 Curvas de Entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'MLP': '#e74c3c', 'RNN': '#3498db', 'LSTM': '#2ecc71', 'GRU': '#9b59b6', 'BiLSTM': '#f39c12'}

# Loss de entrenamiento
ax = axes[0]
for name, hist in all_histories.items():
    ax.plot(hist['train_loss'], label=name, color=colors[name], linewidth=2)
ax.set_title('Train Loss', fontsize=14)
ax.set_xlabel('Época')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy de test
ax = axes[1]
for name, hist in all_histories.items():
    ax.plot(hist['test_acc'], label=name, color=colors[name], linewidth=2)
ax.set_title('Test Accuracy', fontsize=14)
ax.set_xlabel('Época')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# Gap entre train y test (overfitting)
ax = axes[2]
for name, hist in all_histories.items():
    gap = [tr - te for tr, te in zip(hist['train_acc'], hist['test_acc'])]
    ax.plot(gap, label=name, color=colors[name], linewidth=2)
ax.set_title('Overfitting Gap (Train Acc - Test Acc)', fontsize=14)
ax.set_xlabel('Época')
ax.set_ylabel('Gap')
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 Tabla Resumen de Resultados

In [ ]:
print(f"{'Modelo':<12} {'Best Test Acc':>13} {'Final Train Acc':>16} {'Overfitting':>12} {'Tiempo/Época':>13} {'Params':>10}")
print("=" * 80)

model_objects = {'MLP': mlp_model, 'RNN': rnn_model, 'LSTM': lstm_model, 'GRU': gru_model, 'BiLSTM': bilstm_model}

for name, hist in all_histories.items():
    best_test = max(hist['test_acc'])
    final_train = hist['train_acc'][-1]
    overfit = final_train - best_test
    avg_time = np.mean(hist['epoch_time'])
    n_params = sum(p.numel() for p in model_objects[name].parameters() if p.requires_grad)
    print(f"{name:<12} {best_test:>12.3f} {final_train:>15.3f} {overfit:>11.3f} {avg_time:>12.2f}s {n_params:>10,}")

### 6.3 Matrices de Confusión

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))
short_names = [c.split('.')[-1] for c in categories]

all_models = [
    ('MLP', mlp_model, test_loader_tfidf),
    ('RNN', rnn_model, test_loader_seq),
    ('LSTM', lstm_model, test_loader_seq),
    ('GRU', gru_model, test_loader_seq),
    ('BiLSTM', bilstm_model, test_loader_seq),
]

for ax, (name, model, loader) in zip(axes, all_models):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b = X_b.to(device)
            preds = model(X_b).argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(y_b.tolist())
    
    cm = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=short_names)
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(name, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---
## Parte 7: Entendiendo los Gradientes

Una de las motivaciones principales de LSTM y GRU es resolver el **problema de gradientes que desaparecen**.
Vamos a visualizar cómo fluyen los gradientes en cada arquitectura.

In [ ]:
def compute_gradient_norms(model, dataloader, device, max_batches=5):
    """
    Calcula la norma de los gradientes de cada capa del modelo.
    Útil para detectar vanishing/exploding gradients.
    """
    model.train()
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    grad_norms = {}
    
    for i, (X_b, y_b) in enumerate(dataloader):
        if i >= max_batches:
            break
        
        X_b, y_b = X_b.to(device), y_b.to(device)
        model.zero_grad()
        logits = model(X_b)
        loss = criterion(logits, y_b)
        loss.backward()
        
        for name, param in model.named_parameters():
            if param.grad is not None:
                norm = param.grad.norm().item()
                if name not in grad_norms:
                    grad_norms[name] = []
                grad_norms[name].append(norm)
    
    # Promediar
    return {k: np.mean(v) for k, v in grad_norms.items()}


# Calcular gradientes para modelos recurrentes
recurrent_models = {
    'RNN': RNNClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES),
    'LSTM': LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES),
    'GRU': GRUClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES),
}

print("Calculando normas de gradientes (modelos sin entrenar)...\n")
for name, model in recurrent_models.items():
    grads = compute_gradient_norms(model, train_loader_seq, device)
    print(f"\n📊 {name}:")
    for param_name, norm in sorted(grads.items()):
        bar = '█' * min(int(norm * 10), 50)
        print(f"  {param_name:<35} grad_norm: {norm:.6f} {bar}")

**🤔 Preguntas de reflexión:**

1. ¿Cuál modelo tiene los gradientes más uniformes entre capas? ¿Por qué?
2. ¿En la RNN vanilla, cómo se comparan los gradientes de la capa de embedding vs. la capa recurrente?
3. ¿Qué implicaciones tiene esto para el entrenamiento de secuencias largas?

---
## Parte 8: Experimentos — Tu Turno 🔬

Ahora que tienes el pipeline completo, experimenta con las siguientes variaciones.
Registra tus observaciones en cada caso.

### Experimento 1: Efecto del tamaño del hidden state

Entrena un LSTM con diferentes tamaños de hidden state y compara.

In [ ]:
# TODO: Experimenta con diferentes tamaños de hidden_dim
# Prueba: 32, 64, 128, 256
# ¿Cómo cambia la accuracy? ¿Y el overfitting?

hidden_dims_to_try = [32, 64, 128, 256]
exp1_results = {}

for hdim in hidden_dims_to_try:
    print(f"\n--- hidden_dim = {hdim} ---")
    model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, hdim, NUM_CLASSES)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parámetros: {n_params:,}")
    
    # TODO: Entrenar y guardar el resultado
    # hist = train_model(model, train_loader_seq, test_loader_seq, device,
    #                    num_epochs=15, lr=LR, model_name=f'LSTM (h={hdim})')
    # exp1_results[hdim] = hist

# TODO: Graficar los resultados
# Después de entrenar todos, descomenta y ejecuta:
# fig, ax = plt.subplots(figsize=(10, 5))
# for hdim, hist in exp1_results.items():
#     ax.plot(hist['test_acc'], label=f'h={hdim}')
# ax.set_title('Efecto del Hidden Dim en Test Accuracy')
# ax.set_xlabel('Época'); ax.set_ylabel('Accuracy')
# ax.legend(); ax.grid(True, alpha=0.3)
# plt.show()

### Experimento 2: Efecto del Dropout

¿Cuánto dropout es apropiado? ¿Qué pasa sin dropout?

In [ ]:
# TODO: Experimenta con diferentes tasas de dropout
# Prueba: 0.0 (sin dropout), 0.2, 0.3, 0.5
# Observa cómo cambia el gap train/test (overfitting)

dropout_rates = [0.0, 0.2, 0.3, 0.5]
exp2_results = {}

for drop in dropout_rates:
    print(f"\n--- dropout = {drop} ---")
    model = LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, dropout=drop)
    
    # TODO: Entrenar y guardar el resultado
    # hist = train_model(model, train_loader_seq, test_loader_seq, device,
    #                    num_epochs=15, lr=LR, model_name=f'LSTM (drop={drop})')
    # exp2_results[drop] = hist

# TODO: Graficar overfitting gap por dropout rate

### Experimento 3: Capas apiladas (Stacked)

¿Qué pasa si apilamos múltiples capas recurrentes?

In [ ]:
# TODO: Modifica LSTMClassifier para aceptar num_layers
# y experimenta con 1, 2, 3 capas
#
# Pista: nn.LSTM(..., num_layers=N)
# Cuidado: con num_layers > 1, h_n tiene shape (num_layers, batch, hidden)
# Necesitas h_n[-1] para obtener el último layer

class StackedLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, 
                 num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        
        # TODO: Crear LSTM con múltiples capas
        # Nota: cuando num_layers > 1, puedes usar dropout entre capas
        # con el parámetro dropout de nn.LSTM (no aplica a la última capa)
        self.lstm = ...  # <-- Tu código aquí
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (h_n, c_n) = self.lstm(embedded)
        
        # TODO: Extraer hidden state de la ÚLTIMA capa
        h_last = ...  # <-- Tu código aquí (h_n[-1] para la última capa)
        
        logits = self.fc(self.dropout(h_last))
        return logits


# Comparar 1, 2 y 3 capas
for n_layers in [1, 2, 3]:
    model = StackedLSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES, num_layers=n_layers)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"LSTM ({n_layers} capas) - Parámetros: {n_params:,}")

---
## Parte 9: Predicción sobre Texto Nuevo

Veamos cómo usar nuestros modelos entrenados para clasificar texto nuevo.

In [ ]:
@torch.no_grad()
def predict_text(text, model, word2idx, device, is_tfidf=False, tfidf_vectorizer=None):
    """
    Predice la categoría de un texto nuevo.
    """
    model.eval()
    
    if is_tfidf:
        x = tfidf_vectorizer.transform([text]).toarray()
        x = torch.FloatTensor(x).to(device)
    else:
        seq = text_to_sequence(text, word2idx, MAX_LEN)
        x = torch.LongTensor([seq]).to(device)  # Batch de 1
    
    logits = model(x)
    probs = torch.softmax(logits, dim=1)
    pred_class = probs.argmax(dim=1).item()
    confidence = probs[0, pred_class].item()
    
    return categories[pred_class], confidence, probs[0].cpu().numpy()


# Textos de prueba
test_texts = [
    "NASA launched a new satellite to study the surface of Mars and collect geological samples",
    "The pitcher threw a fastball and the batter hit a home run in the ninth inning",
    "The new graphics card supports real-time ray tracing and advanced rendering pipelines",
    "The peace negotiations between the two nations collapsed after the latest conflict",
]

print("Predicciones con BiLSTM:")
print("=" * 80)
for text in test_texts:
    category, confidence, probs = predict_text(text, bilstm_model, word2idx, device)
    print(f"\n📝 \"{text[:80]}...\"")
    print(f"   → Predicción: {category} (confianza: {confidence:.2%})")
    for i, cat in enumerate(categories):
        bar = '█' * int(probs[i] * 30)
        print(f"     {cat:<25} {probs[i]:.2%} {bar}")

---
## Parte 10: Resumen y Reflexiones Finales

**🤔 Responde las siguientes preguntas:**

### Pregunta 1
¿Cuál modelo obtuvo el mejor accuracy en test? ¿Era el esperado? ¿Por qué?

*Tu respuesta:* ...

### Pregunta 2
El MLP con TF-IDF pierde el orden de las palabras. ¿En qué tipo de tareas sería esto un problema grave? ¿En cuáles podría funcionar igual de bien?

*Tu respuesta:* ...

### Pregunta 3
Explica con tus propias palabras por qué la LSTM tiene aproximadamente 4× los parámetros de una RNN vanilla con el mismo `hidden_dim`.

*Tu respuesta:* ...

### Pregunta 4
¿En qué situaciones usarías un modelo bidireccional vs. unidireccional? Da un ejemplo concreto de cada caso.

*Tu respuesta:* ...

### Pregunta 5
En la GRU, la compuerta de update $z_t$ cumple un doble rol. Explica cómo una sola compuerta logra hacer el trabajo de las compuertas forget e input de la LSTM.

*Tu respuesta:* ...

### Tabla Resumen de Arquitecturas

| Aspecto | MLP | RNN | LSTM | GRU | BiLSTM |
|---------|-----|-----|------|-----|--------|
| ¿Preserva orden? | ❌ | ✅ | ✅ | ✅ | ✅ |
| Compuertas | 0 | 0 | 3 (f, i, o) | 2 (r, z) | 3×2 dirs |
| Problema gradientes | N/A | Severo | Resuelto | Resuelto | Resuelto |
| Contexto | Global (BoW) | Pasado → | Pasado → | Pasado → | ← Pasado + Futuro → |
| Uso típico | Baseline | Secuencias cortas | Secuencias largas | Cuando hay restricción de memoria | NER, POS tagging |

---

**Próxima clase:** Transformers y Mecanismos de Atención — ¿Cómo superar las limitaciones de las RNNs?

---
*Laboratorio creado para el curso de NLP y Análisis Semántico — UCB*